# GreenChanger Machine Learning

In [ ]:
import json
import math
import boto3, psycopg
import numpy as np
import pandas as pd
import statsmodels.api as sm
import pickle
from sklearn.model_selection import train_test_split

In [ ]:
HOST = "greenshift-dev.cluster-cl6ugmisezq2.ap-southeast-2.rds.amazonaws.com"
USER = "greenshift_api"

token = boto3.client("rds", region_name="ap-southeast-2").generate_db_auth_token(
    DBHostname=HOST, Port=5432, DBUsername=USER, Region="ap-southeast-2")
conn = psycopg.connect(host=HOST, port=5432, dbname="postgres", user=USER,
                       password=token, sslmode="require")

# define query function to database
def query(sql, params=None):
    with conn.cursor() as cur:
        cur.execute(sql, params)
        cols = [d.name for d in cur.description]
        return pd.DataFrame(cur.fetchall(), columns=cols)

In [ ]:
# get the latest dataset version ids for canopy and tree inventory
canopy_version = query("""
    SELECT DISTINCT dataset_version_id FROM latest_city_canopy_snapshots
    WHERE observed_year = 2021""").iloc[0, 0]
tree_version = query("""
    SELECT DISTINCT dataset_version_id FROM latest_city_melbourne_named_tree_inventory
    """).iloc[0, 0]
canopy_version, tree_version

In [ ]:
SQL = """
WITH hit AS (
  SELECT t.named_tree_id,
         lower(t.scientific_name)     AS species,
         t.year_planted,
         t.precinct,
         p.canopy_snapshot_feature_id AS polygon_id,
         p.calculated_area_m2         AS area_m2
  FROM named_tree_inventory t
  JOIN LATERAL (
    SELECT f.canopy_snapshot_feature_id, f.calculated_area_m2
    FROM canopy_snapshot_feature f
    WHERE f.dataset_version_id = %(canopy_version)s
      AND f.quality_status = 'passed'
      AND ST_Intersects(f.canopy_geometry, t.tree_location)
    LIMIT 1
  ) p ON true
  WHERE t.dataset_version_id = %(tree_version)s
    AND t.quality_status = 'passed'
),
alone AS (
  SELECT polygon_id FROM hit GROUP BY polygon_id HAVING count(*) = 1
)
SELECT h.species, h.year_planted, h.precinct, h.area_m2
FROM hit h JOIN alone USING (polygon_id)
WHERE h.year_planted BETWEEN 2003 AND 2020
"""
df = query(SQL, {"canopy_version": canopy_version, "tree_version": tree_version})
conn.close()
len(df)



In [ ]:
display(df.head(10))

In [ ]:
df["area_m2"] = df["area_m2"].astype(float)
df["age"] = 2021 - df["year_planted"]

df = df[(df["age"] >= 3) & (df["area_m2"] > 0)]
df = df[df["area_m2"] <= 200]
df["log_area"] = np.log(df["area_m2"])

counts = df["species"].value_counts()
df = df[df["species"].isin(counts[counts >= 30].index)]

len(df), df["species"].nunique(), df["age"].min(), df["age"].max()


In [ ]:
# split the dataset into training and testing sets

seed = 42


train, test = train_test_split(df, test_size=0.2, random_state=seed,
                               stratify=df["species"])

len(train), len(test), train["species"].nunique(), test["species"].nunique()

In [ ]:
# quantile regression per species: log(canopy area) = a + b * age
models = {}

# fit p10 / p50 / p90 lines and keep species where all three grow with age
for species, group in train.groupby("species"):
    X = sm.add_constant(group[["age"]])
    fits = {name: sm.QuantReg(group["log_area"], X).fit(q=q)
            for name, q in [("p10", 0.1), ("p50", 0.5), ("p90", 0.9)]}
    if all(f.params["age"] > 0 for f in fits.values()):
        models[species] = fits

len(models)

In [ ]:
def design(ages):
    ages = np.asarray(ages, dtype=float)
    return np.column_stack([np.ones(len(ages)), ages])   # [截距, 树龄]

def predict_range(species, ages):
    X = design(ages)
    a = np.exp(models[species]["p10"].predict(X))
    b = np.exp(models[species]["p90"].predict(X))
    return np.minimum(a, b), np.maximum(a, b)

t = test[test["species"].isin(models)].copy()
for sp, g in t.groupby("species"):
    t.loc[g.index, "lo"], t.loc[g.index, "hi"] = predict_range(sp, g["age"])

t["inside"] = (t["area_m2"] >= t["lo"]) & (t["area_m2"] <= t["hi"])
t["inside"].mean()   # 目标约 0.80

In [ ]:
def pinball(y, yhat, q):
    d = np.asarray(y) - yhat
    return np.mean(np.maximum(q * d, (q - 1) * d))

rows = []
for sp, fits in models.items():
    tr, te = train[train["species"] == sp], test[test["species"] == sp]
    rows.append({
        "species": sp,
        "train_loss_p10": pinball(tr["log_area"], fits["p10"].predict(design(tr["age"])), 0.1),
        "test_loss_p10":  pinball(te["log_area"], fits["p10"].predict(design(te["age"])), 0.1),
        "train_loss_p90": pinball(tr["log_area"], fits["p90"].predict(design(tr["age"])), 0.9),
        "test_loss_p90":  pinball(te["log_area"], fits["p90"].predict(design(te["age"])), 0.9),
    })
loss = pd.DataFrame(rows)
loss.drop(columns="species").mean()

In [ ]:
given_size = {"S": 0, "M": 2, "L": 4} 

for fits in models.values():
    for f in fits.values():
        f.remove_data()

# save the models and metadata to a pickle file
with open("tree_canopy_growth_model.pkl", "wb") as f:
    pickle.dump({
        "models": models,  # model
        "valid_age_range": [3, 17], # valid age range for prediction
        "size_offset_years": given_size,  # size offset years for S, M, L
    }, f)

In [ ]:
# load the exported file back and predict the same way the backend will
with open("tree_canopy_growth_model.pkl", "rb") as f:
    MODEL = pickle.load(f)


def predict_canopy(species, years, size=None, start_width_m=None):
    fits = MODEL["models"].get(species.lower())

    # check if the input parameters are valid
    if fits is None:
        raise ValueError("unsupported species")
    if years < 0:
        raise ValueError("years must be >= 0")
    if (size is None) == (start_width_m is None):
        raise ValueError("give exactly one of size or start_width_m")

    if size is not None:
        if size not in MODEL["size_offset_years"]:
            raise ValueError("size must be S, M or L")
        age0 = float(MODEL["size_offset_years"][size])
    else:
        if start_width_m <= 0:
            raise ValueError("start_width_m must be > 0")
        a50, b50 = fits["p50"].params
        start_area = math.pi * (start_width_m / 2) ** 2
        age0 = max(0.0, (math.log(start_area) - a50) / b50)

    age = age0 + years
    X = np.array([[1.0, age]])
    lo = float(np.exp(fits["p10"].predict(X))[0])
    hi = float(np.exp(fits["p90"].predict(X))[0])
    lo, hi = min(lo, hi), max(lo, hi)

    min_age, max_age = MODEL["valid_age_range"]
    return {
        "canopy_m2_min": round(lo, 1),
        "canopy_m2_max": round(hi, 1),
        "equivalent_age_years": round(age, 1),
        "outside_training_range": not (min_age <= age <= max_age),
    }


species = "platanus x acerifolia"
pd.DataFrame([
    {"input": f"size={s}, years={y}", **predict_canopy(species, y, size=s)}
    for s in MODEL["size_offset_years"] for y in (3, 5, 10)
] + [
    {"input": "start_width_m=3.0, years=5", **predict_canopy(species, 5, start_width_m=3.0)}
])